In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 8
batch_size = 4
vocab_size = 80

In [5]:
with open("./data/wizard_of_oz.txt",'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print(len(text))

232316


In [7]:
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))

81

 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz﻿


In [8]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l] 

In [9]:
print(encode(text[:10]))
print(''.join(decode(encode(text[:10]))))

[80, 1, 1, 28, 39, 42, 39, 44, 32, 49]
﻿  DOROTHY


In [10]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [13]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"For context {context} the target is {target}")

For context tensor([80]) the target is 1
For context tensor([80,  1]) the target is 1
For context tensor([80,  1,  1]) the target is 28
For context tensor([80,  1,  1, 28]) the target is 39
For context tensor([80,  1,  1, 28, 39]) the target is 42
For context tensor([80,  1,  1, 28, 39, 42]) the target is 39
For context tensor([80,  1,  1, 28, 39, 42, 39]) the target is 44
For context tensor([80,  1,  1, 28, 39, 42, 39, 44]) the target is 32


##### <----------------------------------------------------------------------------------------------->
##### get_batch function

In [15]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data)-block_size,size=(batch_size,)) # block_size=8 , batch_size=4
    print(ix)
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

##### <----------------------------------------------------------------------------------------------->

In [18]:
x, y = get_batch('train')
print(f"inputs\n{x}")
print(f"targets\n{y}")

tensor([171386,  78607,  15619,  34035])
inputs
tensor([[73, 62, 66, 58,  9,  1, 73, 68],
        [ 1, 69, 65, 54, 56, 58, 57,  1],
        [61, 58,  1, 54, 62, 71, 11,  0],
        [ 0, 30, 54, 71,  1, 74, 69,  1]], device='cuda:0')
targets
tensor([[62, 66, 58,  9,  1, 73, 68,  1],
        [69, 65, 54, 56, 58, 57,  1, 73],
        [58,  1, 54, 62, 71, 11,  0, 44],
        [30, 54, 71,  1, 74, 69,  1, 62]], device='cuda:0')
